# 2. Solving the Schrödinger Equation II

In Lab 1 you solved the infinite square well with the matrix-element method,
expanding the wave functions in a basis of orthonormalized polynomials. The
energies and wave functions of a quantum system are physical — they must not
depend on the basis you happen to choose (in the limit of infinitely many basis
functions). In this lab you put that idea to work:

1. **A sine basis.** Re-solve the infinite square well using
   $\{\sin(j\pi x/a)\}$ instead of polynomials, and compare.
2. **Wells within wells.** Add a finite step potential inside the box and see how
   the energies and wave functions respond.

The machinery you built in Lab 1 is provided below as a small reusable toolkit so
you can focus on the new physics.

## Setup — the matrix-element toolkit (provided)

Run this cell. It bundles everything from Lab 1 into three functions:

* `gram_schmidt(start_basis)` — orthonormalize a list of SymPy expressions on
  $[0, a]$.
* `hamiltonian_matrix(onb, potential)` — build $H_{ij}$ from an orthonormal basis
  and a potential function `potential(x)`.
* `solve(start_basis, potential)` — do the whole pipeline and return the energies,
  the coefficients, and the orthonormal basis.

You do **not** need to change this cell.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
import sympy as sym
from math import pi as Pi

x = sym.Symbol("x")
a = 1                     # width of the well (natural units: hbar = m = a = 1)

def gram_schmidt(start_basis):
    """Orthonormalize a list of SymPy expressions on [0, a]."""
    def ip(f1, f2):
        return sym.integrate(f1 * f2, (x, 0, a))
    ortho = []
    for f in start_basis:
        for g in ortho:
            f = f - ip(g, f) * g / ip(g, g)
        ortho.append(f)
    return [sym.simplify(f / sym.sqrt(ip(f, f))) for f in ortho]

def hamiltonian_matrix(onb, potential):
    """Hamiltonian matrix H_ij = <phi_i| -1/2 d^2/dx^2 + V |phi_j> on [0, a]."""
    Nb = len(onb)
    f  = [sym.lambdify(x, phi) for phi in onb]
    d2 = [sym.lambdify(x, sym.diff(phi, x, 2)) for phi in onb]
    H = np.zeros([Nb, Nb])
    for j in range(Nb):
        for i in range(j + 1):                      # use H_ij = H_ji
            K = quad(lambda xx: -0.5 * f[i](xx) * d2[j](xx), 0, a)[0]
            V = quad(lambda xx: f[i](xx) * f[j](xx) * potential(xx), 0, a)[0]
            H[i, j] = H[j, i] = K + V
    return H

def solve(start_basis, potential):
    """Return (energies, coefficients, orthonormal_basis)."""
    onb = gram_schmidt(start_basis)
    H = hamiltonian_matrix(onb, potential)
    E, C = np.linalg.eigh(H)
    return E, C.T, onb        # C.T[n] are the coefficients of eigenstate n

def exact_isw_energies(n_max):
    """Exact infinite-square-well energies in natural units: E_n = n^2 pi^2 / 2."""
    return np.array([(n**2) * Pi**2 / 2 for n in range(1, n_max + 1)])

## Warm-up — reproduce the infinite square well

To check the toolkit, re-solve the plain infinite square well ($V = 0$ inside)
with the polynomial basis from Lab 1. The energies should match
$E_n = n^2\pi^2/2$.

In [2]:
Nb = 6

# polynomial basis: x^j (x - a), which vanishes at x = 0 and x = a
poly_basis = [x**j * (x - a) for j in range(1, Nb + 1)]

def free(xx):
    return 0.0            # no potential inside the well

E, C, onb = solve(poly_basis, free)

print("n   numerical      exact")
for n in range(Nb):
    print(f"{n+1:<3} {E[n]:<13.6f} {exact_isw_energies(Nb)[n]:.6f}")

n   numerical      exact
1   4.934802      4.934802
2   19.739237     19.739209
3   44.586812     44.413220
4   79.995958     78.956835
5   175.478386    123.370055
6   285.264806    177.652879


## Exercise 1 — A sine basis

Replace the polynomial basis with a set of sine functions that satisfy the same
boundary conditions $\phi_j(0) = \phi_j(a) = 0$:

$$\left\{ \sin\!\left( \frac{j \pi x}{a} \right) \right\}, \qquad j \geq 1.$$

**(a)** Build `sine_basis` and solve the infinite square well with it. Compare the
energies to the exact values and to the polynomial-basis results from the warm-up.
What do you notice about the accuracy?

**(b)** Look at the expansion coefficients of the eigenstates in each basis (the
rows of `C`). What is different about the coefficients for the sine basis versus
the polynomial basis, and why? *(Hint: the sine functions are already the exact
eigenfunctions of this particular well.)*

In [3]:
# (a) Build the sine basis and solve. Use sym.sin and sym.pi for the symbols.
sine_basis = [sym.sin(j * sym.pi * x / a) for j in range(1, Nb + 1)]
E_sine, C_sine, onb_sine = solve(sine_basis, free)

E_poly = solve(poly_basis, free)[0]
exact = exact_isw_energies(Nb)
print("n    sine           polynomial     exact")
for n in range(Nb):
    print(f"{n+1:<4} {E_sine[n]:<14.8f} {E_poly[n]:<14.8f} {exact[n]:.8f}")

# The sine basis reproduces the exact energies to machine precision for EVERY state,
# because sin(n*pi*x/a) are exactly the eigenfunctions of the infinite square well.
# The polynomial basis is excellent for the ground state but loses accuracy for the
# higher states at fixed Nb.

n    sine           polynomial     exact
1    4.93480220     4.93480222     4.93480220
2    19.73920880    19.73923670    19.73920880
3    44.41321980    44.58681182    44.41321980
4    78.95683521    79.99595777    78.95683521
5    123.37005501   175.47838596   123.37005501
6    177.65287922   285.26480553   177.65287922


In [4]:
# (b) Inspect and compare the coefficients C (polynomial) and C_sine (sine).
E_poly, C_poly, _ = solve(poly_basis, free)
print("|C_sine| (sine basis) - each state is essentially a single basis function:")
print(np.round(np.abs(C_sine), 3))
print("\n|C_poly| (polynomial basis) - each state mixes several basis functions:")
print(np.round(np.abs(C_poly), 3))

# In the sine basis the coefficient matrix is (up to sign) the identity: eigenstate n
# is just sin(n*pi*x/a). In the polynomial basis each eigenstate is a combination of
# several orthonormalized polynomials, so each row has several nonzero entries.

|C_sine| (sine basis) - each state is essentially a single basis function:
[[1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1.]]

|C_poly| (polynomial basis) - each state mixes several basis functions:
[[0.999 0.    0.038 0.    0.001 0.   ]
 [0.    0.991 0.    0.13  0.    0.008]
 [0.037 0.    0.974 0.    0.222 0.   ]
 [0.    0.126 0.    0.941 0.    0.314]
 [0.008 0.    0.222 0.    0.975 0.   ]
 [0.    0.034 0.    0.312 0.    0.949]]


## Exercise 2 — Wells within wells

Now make the problem more interesting. In addition to the infinite walls at
$x = 0$ and $x = a$, add a finite step of height $V_0$ in the middle region
$[b,\ a-b]$ with $b = 0.3$:

$$
V(x)=\left\{\begin{array}{ll}
0   & \quad 0 < x < b,\\
V_0 & \quad b < x < a-b,\\
0   & \quad a-b < x < a.
\end{array}\right.
$$

(The infinite walls are already enforced by the basis, which vanishes at $0$ and
$a$, so the toolkit only needs the finite part of the potential.)

Solve this system with **both** bases and describe how the energies and wave
functions change for:

* a shallow well, $V_0 = -0.1$, and
* a deep well, $V_0 = -10.0$.

How do the two bases compare for each case?

In [5]:
b = 0.3
def step_potential(xx, V0):
    """Finite step of height V0 on [b, a-b], zero elsewhere inside the well."""
    if b < xx < a - b:
        return V0
    return 0.0

for V0 in (-0.1, -10.0):
    pot = lambda xx, V0=V0: step_potential(xx, V0)
    E_poly, _, _ = solve(poly_basis, pot)
    E_sine, _, _ = solve(sine_basis, pot)
    print(f"V0 = {V0}")
    print(" n    polynomial     sine")
    for n in range(Nb):
        print(f" {n+1:<4} {E_poly[n]:<14.6f} {E_sine[n]:.6f}")
    print()

# Shallow well (V0 = -0.1): every level shifts down slightly; the two bases agree to
# several digits. Deep well (V0 = -10): the lowest states drop sharply and localize in
# the central region, and the two bases now disagree more for the higher states -- for a
# stepped potential neither polynomials nor sines are the exact eigenfunctions, so the
# truncation error at fixed Nb is larger.

V0 = -0.1
 n    polynomial     sine
 1    4.864490       4.864489
 2    19.708589      19.708539
 3    44.554722      44.379490
 4    79.953561      78.909280
 5    175.463713     123.330062
 6    285.252233     177.617936



V0 = -10.0
 n    polynomial     sine
 1    -2.453871      -2.455782
 2    16.460142      16.421481
 3    41.713506      41.334082
 4    75.940282      74.345747
 5    174.032757     119.436129
 6    284.037864     174.264909

